# 02 — Train Model + Save Artifact

Creates: `models/fraud_model.joblib`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from joblib import dump
from sklearn.metrics import average_precision_score, precision_recall_fscore_support
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("data/transactions.csv").sort_values(["user_id","ts"]).reset_index(drop=True)
df["ts_dt"] = pd.to_datetime(df["ts"], unit="s")

# expanding stats per user, shifted
df["amt_user_mean_prev"] = df.groupby("user_id")["amount"].expanding().mean().shift(1).reset_index(level=0, drop=True)
df["amt_user_std_prev"]  = df.groupby("user_id")["amount"].expanding().std().shift(1).reset_index(level=0, drop=True)
df["amt_user_mean_prev"] = df["amt_user_mean_prev"].fillna(df["amount"].mean())
df["amt_user_std_prev"]  = df["amt_user_std_prev"].fillna(df["amount"].std() + 1e-6)
df["amount_z_user"] = (df["amount"] - df["amt_user_mean_prev"]) / (df["amt_user_std_prev"] + 1e-6)

def add_roll(g):
    g = g.sort_values("ts_dt").set_index("ts_dt")
    for name, win in {"1h": 3600, "24h": 24*3600}.items():
        w = f"{win}s"
        g[f"txn_count_{name}"] = g["amount"].rolling(w).count().shift(1).fillna(0.0)
        g[f"txn_sum_{name}"]   = g["amount"].rolling(w).sum().shift(1).fillna(0.0)
        g[f"txn_max_{name}"]   = g["amount"].rolling(w).max().shift(1).fillna(0.0)
    return g.reset_index()

df = df.groupby("user_id", group_keys=False).apply(add_roll)
df["is_online"] = (df["channel"]=="online").astype(int)
df["amt_gt_3x_user_mean"] = (df["amount"] > 3.0*df["amt_user_mean_prev"]).astype(int)
df = df.drop(columns=["ts_dt"])

# time split
df = df.sort_values("ts").reset_index(drop=True)
cut = int(len(df)*0.8)
train_df, test_df = df.iloc[:cut], df.iloc[cut:]

numeric = ["amount","device_change","amount_z_user","txn_count_1h","txn_sum_1h","txn_max_1h",
           "txn_count_24h","txn_sum_24h","txn_max_24h","amt_gt_3x_user_mean","is_online"]
categorical = ["merchant_cat","channel"]

X_train = train_df[numeric+categorical]
y_train = train_df["label"].astype(int).values
X_test  = test_df[numeric+categorical]
y_test  = test_df["label"].astype(int).values

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("num", "passthrough", numeric),
])
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
model = Pipeline([("pre", pre), ("clf", clf)]).fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:,1]
pr_auc = float(average_precision_score(y_test, y_prob))

def eval_thr(thr):
    y_pred = (y_prob>=thr).astype(int)
    p,r,f1,_ = precision_recall_fscore_support(y_test,y_pred,average="binary",zero_division=0)
    return float(p),float(r),float(f1)

target_recall = 0.80
chosen = 0.5
for t in np.linspace(0.05,0.95,91):
    p,r,f1 = eval_thr(t)
    if r>=target_recall:
        chosen=float(t); break

metrics = {"pr_auc": pr_auc, "thr_0.5": dict(zip(["precision","recall","f1"], eval_thr(0.5))),
           "thr_tuned": dict(zip(["precision","recall","f1"], eval_thr(chosen))), "chosen_threshold": chosen}
metrics


In [ ]:
Path("models").mkdir(exist_ok=True)
artifact = {"model": model, "numeric_features": numeric, "categorical_features": categorical,
            "threshold": chosen, "metrics": metrics}
dump(artifact, "models/fraud_model.joblib")
"saved models/fraud_model.joblib"
